# Reproducing Natraj et al. (2025) from dandiset 001535

**Paper.** Natraj N, Seko S, Abiri R, Miao R, Yan H, Graham Y, Tu-Chan A, Chang EF, Ganguly K
(2025). Sampling representational plasticity of simple imagined movements across days enables
long-term neuroprosthetic control. *Cell* 188, 1208-1225.
[doi:10.1016/j.cell.2025.02.001](https://doi.org/10.1016/j.cell.2025.02.001). The claims file was
written from the accepted author manuscript in PubMed Central (PMC11932800), with the
supplementary legends; it is not the version of record.

**Dandiset.** [DANDI:001535](https://dandiarchive.org/dandiset/001535/0.260608.0617), version
0.260608.0617. The paper cites no version of the dandiset (its data statement points to OSF for
code and to the lead contact for data). There are two published versions, 0.250714.1218 and
0.260608.0617. Both list the same single asset (`sub-BRAVO1/sub-BRAVO1.nwb`, asset
fa402651-4564-4d67-bb11-941843453978, the same blob, 4773567412 bytes); they differ only in
metadata (a second contributor and the citation), none of which the analysis uses. The most recent
published version is pinned, as the protocol asks when the paper cites none.

**What the dandiset holds.** One NWB file for participant B1: 9737 trials of 256 features at
100 Hz (high-gamma amplitude and low-frequency signals on the 128 channels, the input of the
paper's biLSTM decoder), with a target id and two context flags per trial. It holds no data from
participant B3, no Figure 1 imagined-movement task, no delta, beta and high-gamma band-power
features of the MLP decoders, no session, day, open-loop or closed-loop labels, no online decoder
outputs and no robot kinematics or task outcomes.

| Claims | Tested or blocked |
|---|---|
| rnn-training-data-amount | Tested |
| Figure 1 and S2: fig1g, fig1h, fig1ij, figs2, fig1kl, fig1mn (6) | Blocked, data missing |
| Figures 2, 3, S3 to S6: session counts, decoding accuracies, latent distances and variances, manifold stability, spatial pooling (21) | Blocked, data missing |
| Figure S7: ReFit-KF comparisons (3) | Blocked, data missing |
| Figure 4 and S8A-B: day differences and day decoding (6) | Blocked, data missing or metadata insufficient |
| Figure 5 and S8C-E: plug-and-play performance (6) | Blocked, data missing |
| Figure 6 and S8G-I: IBID kinematics, perturbations, coactivations (4) | Blocked, data missing |
| Figure 7: mode switch and Kinova Jaco tasks (5) | Blocked, data missing |

The reason for each blocked claim is in the attempt record; the data check below prints what the
file holds, which is what the blocks rest on.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import bci

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

started = time.time()
url = bci.asset_url()
meta, table = bci.file_metadata(url)
summ = bci.trial_summaries(url, table.id)
print(f"dandiset {bci.DANDISET_ID} version {bci.DANDISET_VERSION}, {bci.ASSET_PATH}")
print(f"read the metadata and {len(summ)} trials in {time.time() - started:.0f} s")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
for k in ["identifier", "session_description", "session_start_time", "related_publications", "subject"]:
    print(f"{k}: {meta[k]}")
print("top-level groups:", meta["top_level"])
print("processing modules:", meta["processing"], "; analysis:", meta["analysis"],
      "; electrodes table present:", meta["electrodes"])
print("trial table columns:", meta["trial_columns"])
print("trial series attributes:", meta["trial_series_attrs"], "; unit:", meta["trial_series_unit"])
print(f"\ntrials in table: {len(table)}; ids {table.id.min()} to {table.id.max()}, unique: {table.id.is_unique}")
print("every table id has a series acquisition/trial_<id>:", len(summ) == len(table) and (summ.id.values == table.id.values).all())
print("a series after the last id exists:", meta["series_after_last_id"])
print("start_time equals id:", bool((table.start_time == table.id).all()),
      "; stop_time equals id + 1:", bool((table.stop_time == table.id + 1).all()))
print("\ntrials by target_id and context:")
ctx = np.select([table.is_virtual & ~table.is_robot, ~table.is_virtual & table.is_robot,
                 table.is_virtual & table.is_robot], ["virtual", "robot", "both"], "neither")
table["context"] = ctx
print(pd.crosstab(table.target_id.astype(int), table.context, margins=True).to_string())

The file holds one subject, BRAVO1, which is the paper's B1 (male; the paper gives 41 years at
implant, the file P38Y). No participant B3 is in the dandiset. The trial table's start and stop
times are placeholders (the trial id and the id plus one, as their descriptions say, "DUMMY"), and
the session start time is a single date for all trials: nothing in the file gives a trial's day,
session, block, or whether it was open-loop or closed-loop, and there is no electrodes table,
so the channels' places on the 8x16 grid are not given. The dandiset description speaks of 7
directional targets; the table has 9 target ids, 1 to 7 with about 1350 trials each and 8 and 9
with 132 each. Nothing in the file names the targets.

In [ ]:
dur = summ.n_samples / bci.SAMPLING_HZ
print(f"features per sample: {sorted(summ.n_features.unique())}")
print(f"timestamps start at 0 in every trial: {bool((summ.t_first == 0).all())}; "
      f"step from {summ.dt_min.min():.6f} to {summ.dt_max.max():.6f} s (description: 100 Hz)")
print(f"samples per trial: {summ.n_samples.min()} to {summ.n_samples.max()}, median {summ.n_samples.median():.0f}")
print(f"last timestamp equals (samples - 1) / 100 s: {bool(np.allclose(summ.t_last, (summ.n_samples - 1) / 100))}")
print(f"NaN values: {summ.n_nan.sum()}")
print(f"L2 norm of columns 0-127 per trial: {summ.norm_a.min():.6f} to {summ.norm_a.max():.6f}")
print(f"L2 norm of columns 128-255 per trial: {summ.norm_b.min():.6f} to {summ.norm_b.max():.6f}")
print(f"smallest value in columns 0-127 over all trials: {summ.min_a.min():.3g}; in columns 128-255: {summ.min_b.min():.3g}")
print(f"trials with a negative value in columns 0-127: {(summ.min_a < 0).sum()} of {len(summ)}; "
      f"in columns 128-255: {(summ.min_b < 0).sum()}")
print(f"largest absolute value over all trials: {summ.abs_max.max():.3g}")
dup = summ[summ.duplicated("hash", keep=False)].sort_values("hash")
print(f"\ntrials whose data are byte-identical to another trial: {len(dup)}")
if len(dup):
    print(dup.groupby("hash").id.apply(list).to_string())
means = np.round(np.vstack(summ["mean"].values), 7)
_, inv, counts = np.unique(means, axis=0, return_inverse=True, return_counts=True)
print(f"trials sharing their 256 time-averaged feature values with another trial: {(counts[inv] > 1).sum()}")

Every trial has 256 features sampled every 0.01 s, as described (100 Hz), and lasts 1.8 to
11.1 s. The first 128 columns have no negative value in 9239 of the 9737 trials (the smallest value
is -0.015) and the last 128 take both signs in every trial, which fits high-gamma amplitude followed by the low-frequency signal; the description names both but not the order, so this order
is consistent with the data, not shown. Each block of 128 columns has an L2 norm of 1 over the
whole trial, which is what the Methods (Spatiotemporal feature extraction for RNNs) describe for
the biLSTM input: "each feature data matrix was transformed to have an L 2 norm of 1 and
concatenated together". The values are therefore not in physical units and are scaled per trial;
the unit "arbitrary" says so. The duplicate checks above count trials deposited more than once.

### Do the trial labels belong to the trial data?

There are no event times inside a trial and no session clock, so the check is that each trial's
target id belongs to the series of the same id. Each trial is reduced to its 256 time-averaged
features and the target is decoded by cross-validated logistic regression (5 stratified folds, a
standard choice). The contrast is the same decoder with the labels shifted by one trial, and
with the labels shuffled.

In [ ]:
rng = np.random.default_rng(0)
X = np.vstack(summ["mean"].values).astype(float)
y = table.target_id.astype(int).values
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
cv = StratifiedKFold(5, shuffle=True, random_state=0)
rows = []
for name, labels in [("true labels", y), ("shifted by +1 trial", np.roll(y, 1)),
                     ("shifted by -1 trial", np.roll(y, -1)), ("shuffled", rng.permutation(y))]:
    acc = cross_val_score(clf, X, labels, cv=cv)
    rows.append({"labels": name, "accuracy": acc.mean(), "fold_min": acc.min(), "fold_max": acc.max()})
chance = pd.Series(y).value_counts(normalize=True).max()
print(pd.DataFrame(rows).round(4).to_string(index=False))
print(f"chance (largest class share): {chance:.4f}; 1/9 = {1/9:.4f}")
by_ctx = []
for c in ["virtual", "robot"]:
    m = (table.context == c).values
    acc = cross_val_score(clf, X[m], y[m], cv=cv).mean()
    by_ctx.append({"context": c, "trials": int(m.sum()), "accuracy": round(acc, 4)})
print(pd.DataFrame(by_ctx).to_string(index=False))

With the true labels the target is decoded from the trials' own features with 0.72 accuracy
(0.72 in virtual and 0.76 in robot trials), against 0.17 to 0.18 with the labels moved by one
trial, 0.13 with them shuffled, and 0.14 for always naming the largest class. The target ids are
consistent with the series they share an id with, and not with the neighbouring trials' series. This shows the table and the series are
aligned trial by trial; it cannot show anything about timing within a trial, as the file has no
event times.

## rnn-training-data-amount

> The biLSTM decoder was trained on about 14 hours and 48 minutes of open-loop and closed-loop
> data from B1's virtual and real-world hDoF experiments, before augmentation and segmentation.

Expected 888 min, tolerance 88.8 min. The criterion: the total duration of the trials used to
train the RNN, before segmentation; reading (a) the whole stored trial, reading (b) the active
portion only; shown with and without the real-robot trials when the dataset labels contexts.

A trial's duration is its number of samples divided by 100 Hz. The dandiset's description says
its trials are the ones "uploaded for the development and testing of novel BCI decoding
architectures" and points to the paper's biLSTM; it does not say that they are exactly the
paper's training set. Reading (b) cannot be run, because the file has no task states, but the
active portion of a trial is part of the stored trial, so reading (b) is at most reading (a).

In [ ]:
dur = summ.n_samples / bci.SAMPLING_HZ
d = table.assign(duration_s=dur.values)
rows = [{"trials": "all", "n": len(d), "total_min": d.duration_s.sum() / 60}]
for c in ["virtual", "robot"]:
    s = d[d.context == c]
    rows.append({"trials": c, "n": len(s), "total_min": s.duration_s.sum() / 60})
s = d[d.target_id <= 7]
rows.append({"trials": "targets 1 to 7 only", "n": len(s), "total_min": s.duration_s.sum() / 60})
alt = ((summ.n_samples - 1) / bci.SAMPLING_HZ).sum() / 60
rows.append({"trials": "all, last minus first timestamp", "n": len(d), "total_min": alt})
res = pd.DataFrame(rows)
res["total_h_min"] = res.total_min.map(lambda m: f"{int(m // 60)} h {m % 60:.1f} min")
res["within_888_pm_88.8"] = (res.total_min - 888).abs() <= 88.8
print(res.round(1).to_string(index=False))
print(f"\nexpected 888 min; the shortest duration that would match is {888 - 88.8:.1f} min")
print(f"trial duration: {d.duration_s.min():.2f} to {d.duration_s.max():.2f} s, median {d.duration_s.median():.2f} s")
print(d.groupby("context").duration_s.describe().round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
for c in ["virtual", "robot"]:
    ax.hist(d.duration_s[d.context == c], bins=np.arange(0, 12.01, 0.1), alpha=0.6, label=c)
ax.set_xlabel("trial duration (s)")
ax.set_ylabel("trials")
ax.legend()
plt.tight_layout()
plt.show()
print(f"all {len(d)} trials are inside the axes (0 to 12 s); longest {d.duration_s.max():.2f} s")

**Result: reproduced.** Under reading (a), the whole stored trial, the 9737 trials last 837.4 min
(13 h 57.4 min), within the 888 +/- 88.8 min of the claim (from 799.2 min); without targets 8 and
9 they last 814.3 min, also within. Reading (b), the active portion, cannot be run, as the file
has no task states; it can only be shorter than (a). The 7851 virtual trials alone last 540.0 min
and the 1886 robot trials 297.3 min. The paper counts its 14 h 48 min over the virtual environment
and "practice with the Kinova Jaco in the real-world using the MLP", so the robot trials belong in
the total if they are that practice; the file does not say whether they are that practice or the
later fine-tuning data. Without them the total would be outside the tolerance.

## Summary

| Claim | Outcome |
|---|---|
| rnn-training-data-amount | Reproduced |
| The other 51 claims | Blocked (data missing, or metadata insufficient for the Figure 4 claims on B1 alone) |

How the analysis differs from the paper's: the one tested claim is a count of stored data, and
the dandiset does not say whether its trials are the whole of the paper's training data or how
much of each trial was kept.

What was found about the dataset:

- It holds only participant B1's biLSTM input features (high gamma and low frequency, 100 Hz, L2
  normalized per trial), with a target id and a virtual or robot flag per trial. Participant B3,
  the Figure 1 task, the band-power features of the MLP decoders, online decoder outputs and robot
  kinematics are not in it.
- No trial has a day, session, block or open-loop or closed-loop label; trial start and stop times
  are placeholders.
- The target ids carry no names, and there are 9 of them where the description says 7 targets.
- There is no electrodes table, so the channels cannot be placed on the grid.
- The robot trials (1886, 9 to 11 s each) are not said to be either the MLP practice the paper
  counts in its 14 h 48 min or the later fine-tuning of the RNN.
- No trial is deposited twice (no two trials have identical data or identical time-averaged
  features).